# 智能客服机器人

## LangChain 智能客服机器人

本篇将前面学到的知识整合起来，构建一个完整的智能客服机器人。它能够查询知识库、处理订单、在必要时转接人工。

---

## 需求分析

| 功能 | 实现方式 |
| --- | --- |
| 知识库问答 | RAG 检索 + 模型回答 |
| 订单查询 | @tool 工具函数 |
| 对话记忆 | SqliteSaver Checkpointer |
| 敏感内容过滤 | @before\_model Middleware |
| 人工转接 | HITL interrupt() |

---

## 完整代码

## 实例

\# 文件路径：customer\_service\_bot.py  
\# pip install langchain langchain-deepseek langchain-chroma chromadb  
from dotenv import load\_dotenv  
load\_dotenv()  
  
from typing import Annotated  
from langchain.tools import tool  
from langchain.agents import create\_agent  
from langchain.agents.middleware import before\_model, after\_model, wrap\_tool\_call  
from langchain.chat\_models import init\_chat\_model  
from langchain.messages import HumanMessage  
from langchain\_openai import OpenAIEmbeddings  
from langchain\_chroma import Chroma  
from langchain\_text\_splitters import RecursiveCharacterTextSplitter  
from langgraph.checkpoint.sqlite import SqliteSaver  
from langgraph.types import interrupt, Command  
  
\# ========== 1. 准备知识库 ==========  
  
knowledge\_base = \[  
"菜鸟教程 RUNOOB 创立于 2013 年，是国内领先的免费编程学习平台。",  
"平台提供 300+ 套教程，涵盖 Python、Java、HTML、CSS、JavaScript 等。",  
"Python3 基础教程共 30 章，累计学习人次超 500 万。课程完全免费。",  
"VIP 会员费用为 ¥99/月，¥799/年，包含视频课程和一对一答疑服务。",  
"退款政策：购买 7 天内且在 3 节课以内可全额退款。",  
"平台支持在线编程环境，无需安装任何软件即可编写运行代码。",  
"客服工作时间：周一至周五 9:00-18:00，周末 10:00-16:00。",  
\]  
  
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  
chunks = RecursiveCharacterTextSplitter(chunk\_size=200, chunk\_overlap=30  
).create\_documents(knowledge\_base)  
vector\_store = Chroma.from\_documents(chunks, embeddings)  
retriever = vector\_store.as\_retriever(search\_kwargs={"k": 3})  
  
\# ========== 2. 定义工具 ==========  
  
@tool  
def search\_kb(query: str) -> str:  
"""搜索菜鸟教程知识库，获取关于平台、课程、政策等官方信息。  
  
Args:  
query: 搜索问题或关键词  
"""  
docs = retriever.invoke(query)  
if not docs:  
return "未找到相关信息，建议转接人工客服。"  
return "\\n".join(f"- {doc.page\_content}" for doc in docs)  
  
\# 模拟订单数据库  
orders\_db = {  
"ORD-2024-001": {"user": "小明", "item": "VIP 年费会员",  
"amount": 799, "status": "已完成", "date": "2024-01-15"},  
"ORD-2024-002": {"user": "小明", "item": "Python 实战课程",  
"amount": 199, "status": "配送中", "date": "2024-03-20"},  
}  
  
@tool  
def query\_order(order\_id: str) -> str:  
"""根据订单号查询订单状态和详情。  
  
Args:  
order\_id: 订单号，如 ORD-2024-001  
"""  
order = orders\_db.get(order\_id.upper())  
if not order:  
return f"未找到订单 {order\_id}。请确认订单号是否正确。"  
return (f"订单 {order\_id}：{order\['item'\]} | "  
f"金额 ¥{order\['amount'\]} | "  
f"状态 {order\['status'\]} | "  
f"日期 {order\['date'\]}")  
  
@tool  
def transfer\_to\_human(reason: str) -> str:  
"""将用户转接给人工客服。  
  
Args:  
reason: 转接原因  
"""  
approval = interrupt({  
"action": "transfer\_to\_human",  
"reason": reason,  
"message": f"用户请求转接人工客服，原因：{reason}。是否转接？"  
})  
if approval.get("confirmed"):  
return (f"已为您转接人工客服，预计等待 {approval.get('wait\_time', 3)} 分钟。"  
f"工单号：TK-{approval.get('ticket\_id', 'N/A')}")  
return "转接已取消，我继续为您服务。"  
  
\# ========== 3. 定义 Middleware ==========  
  
@before\_model  
def content\_guard(state, runtime):  
"""过滤用户输入中的不当内容"""  
last\_msg = state\["messages"\]\[-1\] if state.get("messages") else None  
if not last\_msg:  
return None  
content = str(getattr(last\_msg, 'content', ''))  
blocked = \["黄X", "X博", "违法"\]  
for word in blocked:  
if word in content:  
return {  
"jump\_to": "end",  
"messages": \[HumanMessage(content="抱歉，我不能处理这个请求。")\]  
}  
return None  
  
@after\_model  
def auto\_signature(state, runtime):  
"""自动追加客服签名"""  
msgs = state.get("messages", \[\])  
if not msgs:  
return None  
last = msgs\[-1\]  
if last.type == "ai" and last.content and not (  
hasattr(last, 'tool\_calls') and last.tool\_calls  
):  
from langchain.messages import AIMessage  
return {"messages": \[AIMessage(  
content=last.content  
\+ "\\n\\n---\\n菜鸟教程 RUNOOB 客服中心 | 工作时间 9:00-18:00"  
)\]}  
return None  
  
\# ========== 4. 创建 Agent ==========  
  
checkpointer = SqliteSaver.from\_conn\_string("customer\_service.db")  
model = init\_chat\_model("deepseek:deepseek-v4-flash", temperature=0)  
  
agent = create\_agent(  
model=model,  
tools=\[search\_kb, query\_order, transfer\_to\_human\],  
middleware=\[content\_guard, auto\_signature\],  
checkpointer=checkpointer,  
system\_prompt="""你是菜鸟教程 RUNOOB 的智能客服"小菜"。  
  
\## 你的职责  
1\. 热情接待每一位用户，用"您"称呼  
2\. 关于平台信息、课程内容、政策等问题，使用 search\_kb 查询  
3\. 关于订单查询，使用 query\_order 工具  
4\. 遇到无法解决的问题，使用 transfer\_to\_human 转接人工  
  
\## 行为准则  
\- 回答简洁，每次 2-3 句话  
\- 不知道的就查询知识库，查不到就诚实告知  
\- 保持友好亲切的语气""",  
)  
  
\# ========== 5. 对话接口 ==========  
  
def chat(thread\_id: str, message: str) -> str:  
"""处理用户消息并返回回复"""  
config = {"configurable": {"thread\_id": thread\_id}}  
  
\# 运行 Agent  
result = agent.invoke(  
{"messages": \[HumanMessage(content=message)\]},  
config=config,  
)  
  
\# 检查是否需要转接（HITL）  
state = agent.get\_state(config)  
if state.tasks and state.tasks\[0\].interrupts:  
interrupt\_info = state.tasks\[0\].interrupts\[0\].value  
return f"\[需要审批\] {interrupt\_info.get('message', '')}"  
  
return result\["messages"\]\[-1\].content  
  
\# ========== 6. 测试 ==========  
  
if \_\_name\_\_ == "\_\_main\_\_":  
user\_id = "user\_xiaoming"  
  
print("=== 测试 1：知识库查询 ===")  
print(chat(user\_id, "Python3 教程有多少章？"))  
print()  
  
print("=== 测试 2：订单查询 ===")  
print(chat(user\_id, "我的订单 ORD-2024-001 状态是什么？"))  
print()  
  
print("=== 测试 3：VIP 咨询 ===")  
print(chat(user\_id, "VIP 会员多少钱？"))  
print()  
  
print("=== 测试 4：测试记忆 ===")  
print(chat(user\_id, "我刚才问过什么问题？"))

运行结果：

In [ ]:
=== 测试 1：知识库查询 ===
根据知识库信息，Python3 基础教程共 30 章，累计学习人次超过 500 万，课程完全免费。
---
菜鸟教程 RUNOOB 客服中心 | 工作时间 9:00-18:00

=== 测试 2：订单查询 ===
您的订单 ORD-2024-001：VIP 年费会员，金额 ¥799，状态 已完成，日期 2024-01-15。
---
菜鸟教程 RUNOOB 客服中心 | 工作时间 9:00-18:00

=== 测试 3：VIP 咨询 ===
VIP 会员费用为 ¥99/月或 ¥799/年，包含视频课程和一对一答疑服务。
---
菜鸟教程 RUNOOB 客服中心 | 工作时间 9:00-18:00

=== 测试 4：测试记忆 ===
您刚才咨询了 Python3 教程章节数、订单 ORD-2024-001 的状态，以及 VIP 会员的价格。
还有什么需要帮您的吗？
---
菜鸟教程 RUNOOB 客服中心 | 工作时间 9:00-18:00

---

## 项目总结

这个客服机器人整合了以下 LangChain 特性：

| 特性 | 在项目中的使用 |
| --- | --- |
| RAG 检索 | search\_kb 工具 + Chroma 向量存储 |
| 工具调用 | query\_order、transfer\_to\_human |
| Checkpointer | SqliteSaver 持久化对话，实现多轮记忆 |
| Middleware | before\_model 内容过滤 + after\_model 签名追加 |
| HITL | interrupt() 实现人工转接审批 |